# 34 · E5 Visibility / Leakage Grid · 真器件

> ⚠️ 真连 B1500，接 pFeFET 器件。前置：E1 (30) PASS。

**目的 (H7 Svis, H9 Leakage)**：
- 检验 MW 正负是否随读条件变化（Vg_read × Vd_read 网格）
- 若 MW 在某 Vd 下为正而在另一 Vd 下为负 → Svis（跨导可见度）主导
- 大 Vd 下 MW 更正 → 沟道主导；大 Vd 下 MW 更负 → 漏电/GIDL 主导

序列：Reset → Write → delay(10μs or 1s) → single-point read at (Vg, Vd)

网格：Vg ∈ {−0.4, −0.2, 0, +0.2, +0.4}V × Vd ∈ {0.01, 0.05, 0.10, 0.50}V

测试人：**yhzang**

In [ ]:
import sys, os, time, random, datetime, itertools
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fefetlab.measurements.wgfmu import (
    ensure_wgfmu_dll_path, autodetect_visa_addr,
    autodetect_wgfmu_chan, RealWgfmuBackend,
)

print("Python:", sys.version.split()[0])
print("project root:", ROOT)

In [ ]:
dll = ensure_wgfmu_dll_path()
print(f"✅ wgfmu.dll: {dll}")

VISA_ADDR = autodetect_visa_addr("B1500")
print(f"✅ B1500 VISA addr: {VISA_ADDR}")

backend = RealWgfmuBackend()
backend.load()
backend.open_session(VISA_ADDR)
backend.set_timeout(30.0)
channel_ids = backend.get_channel_ids()
print(f"detected channels: {channel_ids}")

GATE_CH  = autodetect_wgfmu_chan(backend, prefer=201)
DRAIN_CH = 202
assert DRAIN_CH in channel_ids
print(f"✅ gate={GATE_CH}, drain={DRAIN_CH}")
backend.close_session()

In [ ]:
# ── USER-EDITABLE PARAMETERS ──────────────────────────────────
QUICK_MODE  = True    # True: Vd=[0.05,0.50], delays=[10us], 2 reps; False: full 5×4 grid

DEVICE_ID   = "dev001"
GEOMETRY    = "W5L10"

V_ERS       = +5.0
V_PGM       = -5.0
T_WRITE     = 100e-6
T_RISE_FALL = 100e-9
T_RESET     = 1e-3

# Read grid
VG_GRID_FULL  = [-0.4, -0.2, 0.0, +0.2, +0.4]
VD_GRID_QUICK = [0.05, 0.50]
VD_GRID_FULL  = [0.01, 0.05, 0.10, 0.50]

DELAYS_QUICK = [10e-6]
DELAYS_FULL  = [10e-6, 1.0]

T_READ      = 5e-6
N_PTS_READ  = 5

N_REPS_QUICK = 2
N_REPS_FULL  = 5
RANDOM_SEED  = 42
# ── END PARAMETERS ────────────────────────────────────────────

vg_grid  = VG_GRID_FULL
vd_grid  = VD_GRID_QUICK if QUICK_MODE else VD_GRID_FULL
delays   = DELAYS_QUICK  if QUICK_MODE else DELAYS_FULL
N_REPS   = N_REPS_QUICK  if QUICK_MODE else N_REPS_FULL
TAG      = "quick" if QUICK_MODE else "full"

# Each shot: write + delay + read at [all Vg] at one fixed Vd
# → outer loops: state × delay × Vd, inner: multiple Vg per shot
outer_combos = list(itertools.product(["ERS", "PGM"], delays, vd_grid))
total_shots  = len(outer_combos) * N_REPS

print(f"Mode  : {TAG}")
print(f"Grid  : {len(vg_grid)} Vg × {len(vd_grid)} Vd")
print(f"Outer : {len(outer_combos)} = 2 states × {len(delays)} delays × {len(vd_grid)} Vd")
print(f"Total shots: {total_shots} (each returns {len(vg_grid)} Vg readings)")

In [ ]:
def _run_shot(
    backend, gate_ch, drain_ch,
    v_write, t_write, t_rise_fall, t_reset, t_delay,
    vg_reads, vd_read, t_read, n_pts=5,
) -> list:
    """Write-delay-read shot with arbitrary vg_reads list and vd_read.

    Used for E5 grid scans: pass vg_reads=vg_grid, vd_read varies per shot.
    Returns list of per-Vg_read dicts.
    """
    T_GAP  = 100e-9
    GUARD  = 200e-9
    t_delay = max(t_delay, 200e-9)

    backend.clear()

    backend.create_pattern("gp", 0.0)
    backend.add_vector("gp", t_reset,     0.0)
    backend.add_vector("gp", t_rise_fall, float(v_write))
    backend.add_vector("gp", t_write,     float(v_write))
    backend.add_vector("gp", t_rise_fall, 0.0)
    backend.add_vector("gp", t_delay,     0.0)

    t_pre = t_reset + t_rise_fall + t_write + t_rise_fall + t_delay
    backend.create_pattern("dp", 0.0)
    backend.add_vector("dp", t_pre, 0.0)

    meas_win = t_read - GUARD
    interval = meas_win / max(n_pts, 1)
    average  = interval * 0.8

    t_cursor = t_pre
    read_t0s = []
    for i, vg in enumerate(vg_reads):
        t_cursor += t_rise_fall
        read_t0s.append(t_cursor)
        t_cursor += t_read + t_rise_fall
        backend.add_vector("gp", t_rise_fall, float(vg))
        backend.add_vector("gp", t_read,       float(vg))
        backend.add_vector("gp", t_rise_fall,  0.0)
        backend.add_vector("dp", t_rise_fall, vd_read)
        backend.add_vector("dp", t_read,       vd_read)
        backend.add_vector("dp", t_rise_fall,  0.0)
        if i < len(vg_reads) - 1:
            t_cursor += T_GAP
            backend.add_vector("gp", T_GAP, 0.0)
            backend.add_vector("dp", T_GAP, 0.0)

    for i in range(len(vg_reads)):
        t_ev = read_t0s[i] + GUARD
        backend.set_measure_event("gp", f"g{i}", t_ev, n_pts, interval, average, "averaged")
        backend.set_measure_event("dp", f"d{i}", t_ev, n_pts, interval, average, "averaged")

    backend.add_sequence(gate_ch,  "gp", 1)
    backend.add_sequence(drain_ch, "dp", 1)

    backend.initialize()
    for ch in [gate_ch, drain_ch]:
        backend.set_operation_mode(ch, "FASTIV")
        backend.set_force_voltage_range(ch, "AUTO")
        backend.set_measure_enabled(ch, True)
        backend.set_measure_mode(ch, "CURRENT")
        backend.set_measure_current_range(ch, "1MA")
    backend.connect(gate_ch)
    backend.connect(drain_ch)

    backend.execute()
    backend.wait_until_completed()

    g_df = backend.get_measure_values(gate_ch)
    d_df = backend.get_measure_values(drain_ch)
    backend.disconnect(gate_ch)
    backend.disconnect(drain_ch)

    g_t = g_df["time_s"].values;  g_v = g_df["value"].values
    d_t = d_df["time_s"].values;  d_v = d_df["value"].values

    results = []
    for i, vg in enumerate(vg_reads):
        t0 = read_t0s[i] + GUARD
        t1 = t0 + meas_win
        d_sub = d_v[(d_t >= t0) & (d_t <= t1)]
        g_sub = g_v[(g_t >= t0) & (g_t <= t1)]
        results.append(dict(
            Vg_read_V=float(vg), Vd_read_V=vd_read,
            Id_mean_A = float(np.nanmean(d_sub)) if len(d_sub) > 0 else float("nan"),
            Id_std_A  = float(np.nanstd(d_sub))  if len(d_sub) > 1 else float("nan"),
            Ig_mean_A = float(np.nanmean(g_sub)) if len(g_sub) > 0 else float("nan"),
            n_d=int(len(d_sub)), n_g=int(len(g_sub)),
        ))
    return results


print("✅ _run_shot() defined")

In [ ]:
ts_start = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_dir  = ROOT / "runs" / f"{ts_start}_E5_visibility_{TAG}"
run_dir.mkdir(parents=True, exist_ok=True)
out_csv  = run_dir / "visibility_results.csv"
print(f"Output : {run_dir}")

random.seed(RANDOM_SEED)
outer_run = list(outer_combos)
random.shuffle(outer_run)
print(f"Shuffled {len(outer_run)} outer combos")

In [ ]:
all_rows = []
seq_id   = 0

backend.open_session(VISA_ADDR)
t_exp0 = time.time()
try:
    print(f"Starting E5 Visibility: {total_shots} shots × {len(vg_grid)} Vg each")

    for rep in range(N_REPS):
        for state, t_delay, vd in outer_run:
            v_write = V_ERS if state == "ERS" else V_PGM
            shot_timeout = max(30.0, t_delay * 3 + 10.0)
            backend.set_timeout(shot_timeout)

            ts_iso = datetime.datetime.now().isoformat(timespec="seconds")
            t_sh   = time.time()
            note   = ""
            try:
                rr = _run_shot(
                    backend, GATE_CH, DRAIN_CH,
                    v_write=v_write, t_write=T_WRITE,
                    t_rise_fall=T_RISE_FALL, t_reset=T_RESET,
                    t_delay=t_delay, vg_reads=vg_grid,
                    vd_read=vd, t_read=T_READ, n_pts=N_PTS_READ,
                )
            except Exception as exc:
                note = f"ERR:{exc}"
                print(f"  !! {state} d={t_delay:.1e} Vd={vd}: {exc}")
                rr = [dict(Vg_read_V=vg, Vd_read_V=vd,
                           Id_mean_A=float("nan"), Id_std_A=float("nan"),
                           Ig_mean_A=float("nan"), n_d=0, n_g=0)
                      for vg in vg_grid]
                for ch in [GATE_CH, DRAIN_CH]:
                    try: backend.disconnect(ch)
                    except Exception: pass
                try: backend.close_session()
                except Exception: pass
                time.sleep(1.0)
                backend.open_session(VISA_ADDR)

            for r in rr:
                all_rows.append(dict(
                    timestamp_iso=ts_iso, device_id=DEVICE_ID,
                    geometry=GEOMETRY, sequence_id=seq_id,
                    repeat_index=rep, state_target=state,
                    delay_s=t_delay,
                    Vg_read_V=r["Vg_read_V"], Vd_read_V=r["Vd_read_V"],
                    Id_mean_A=r["Id_mean_A"], Id_std_A=r["Id_std_A"],
                    Ig_mean_A=r["Ig_mean_A"], note=note,
                ))
            seq_id += 1
            print(f"  rep={rep} {state:3s} d={t_delay:.1e} Vd={vd:.2f}V | "
                  f"{len(vg_grid)} Vg pts | {time.time()-t_sh:.2f}s")

        pd.DataFrame(all_rows).to_csv(out_csv, index=False, encoding="utf-8-sig")

    print(f"\n✅ Done in {time.time()-t_exp0:.1f}s. Saved: {out_csv}")

except Exception as exc:
    print(f"\n❌ Failed: {exc}")
    raise
finally:
    for ch in [GATE_CH, DRAIN_CH]:
        try: backend.disconnect(ch)
        except Exception: pass
    backend.close_session()

df = pd.DataFrame(all_rows)
print(df.shape)

In [ ]:
df = pd.read_csv(out_csv)

vg_u  = sorted(df["Vg_read_V"].unique())
vd_u  = sorted(df["Vd_read_V"].unique())
del_u = sorted(df["delay_s"].unique())

n_row = len(del_u)
fig, axes = plt.subplots(n_row, len(vd_u), figsize=(4*len(vd_u), 4*n_row), squeeze=False)

for row, t_del in enumerate(del_u):
    for col, vd in enumerate(vd_u):
        ax = axes[row, col]
        sub = df[(df["delay_s"] == t_del) & (df["Vd_read_V"] == vd)]
        for state, color in [("ERS", "#1f77b4"), ("PGM", "#d62728")]:
            s = sub[sub["state_target"] == state]
            g = s.groupby("Vg_read_V")["Id_mean_A"].mean() * 1e9
            ax.plot(g.index, g.values, "o-", color=color, label=state, ms=5)
        ax.axhline(0, color="gray", lw=0.5)
        ax.set_xlabel("Vg_read (V)")
        ax.set_ylabel("Id (nA)")
        ax.set_title(f"Vd={vd:.2f}V, delay={t_del*1e6:.0f}μs")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

plt.suptitle(f"E5 Visibility – Id vs Vg_read | {DEVICE_ID} {GEOMETRY}")
plt.tight_layout()
plt.savefig(run_dir / "visibility_id_vs_vg.png", dpi=150, bbox_inches="tight")
plt.show()

# MW(Vg, Vd) heatmap at each delay
for t_del in del_u:
    sub = df[df["delay_s"] == t_del]
    ers_g = sub[sub["state_target"]=="ERS"].groupby(["Vg_read_V","Vd_read_V"])["Id_mean_A"].mean()
    pgm_g = sub[sub["state_target"]=="PGM"].groupby(["Vg_read_V","Vd_read_V"])["Id_mean_A"].mean()
    mw    = (ers_g - pgm_g).rename("MW_A").reset_index()
    pivot = mw.pivot(index="Vd_read_V", columns="Vg_read_V", values="MW_A") * 1e9

    vmax = max(abs(pivot.values[~np.isnan(pivot.values)].max()),
               abs(pivot.values[~np.isnan(pivot.values)].min()))
    fig2, ax2 = plt.subplots(figsize=(7, 3))
    im = ax2.imshow(pivot.values, aspect="auto", cmap="RdBu",
                    vmin=-vmax, vmax=vmax, origin="lower")
    ax2.set_xticks(range(len(pivot.columns)))
    ax2.set_xticklabels([f"{v:+.1f}" for v in pivot.columns])
    ax2.set_yticks(range(len(pivot.index)))
    ax2.set_yticklabels([f"{v:.2f}" for v in pivot.index])
    ax2.set_xlabel("Vg_read (V)")
    ax2.set_ylabel("Vd_read (V)")
    ax2.set_title(f"MW (nA) | delay={t_del*1e6:.0f}μs | {DEVICE_ID}")
    plt.colorbar(im, ax=ax2, label="MW (nA)")
    for r in range(pivot.shape[0]):
        for c in range(pivot.shape[1]):
            v = pivot.values[r, c]
            ax2.text(c, r, f"{v:.0f}" if not np.isnan(v) else "?",
                     ha="center", va="center", fontsize=8)
    plt.tight_layout()
    plt.savefig(run_dir / f"visibility_mw_heatmap_d{t_del:.0e}.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    print(f"delay={t_del:.1e}s MW heatmap (nA):")
    print(pivot.round(1))

## 通过标准

- [ ] 无硬件错误，CSV 完整（shots × Vg_grid 行）
- [ ] MW 热图有意义的 Vd/Vg 依赖
- [ ] 若 MW 在 Vd=0.05V 和 Vd=0.50V 下符号不同 → Svis 是主导机制
- [ ] Ig_mean < 1μA 所有点（无栅漏异常）

**结论记录**：MW 符号翻转发生在 Vd ≈ ___ V，Svis 机制确认/否定：___